# AirSense — Localized AQI Prediction & Health Advisory System
### Empirical ML Pipeline Addressing 6 Documented Literature Gaps

This notebook documents the end-to-end data exploration, cleaning, chronological model training, stacking ensemble development, and error analysis across 26 Indian cities.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
sys.path.append(os.path.abspath('..'))

sns.set_theme(style='whitegrid')
print('Libraries imported successfully!')

## 1. Load Raw CPCB Dataset & Inspect Initial Missingness

In [ ]:
from src.data_loader import load_raw_data
df_raw = load_raw_data()
print('Dataset Shape:', df_raw.shape)
df_raw.head()

## 2. Address Gap 5: Sensor Dropout ('Fake Zero' Detection) & Data Cleaning

In [ ]:
from src.data_cleaner import clean_and_prepare_data
df_clean, quality_summary = clean_and_prepare_data()
print(json.dumps(quality_summary, indent=2))

## 3. Address Gap 4 & Gap 1: Chronological Splitting & Model Benchmarking
- Linear Regression (Baseline 1)
- Decision Tree (Baseline 2)
- Random Forest (Tuned Ensemble 1)
- Gradient Boosting (Tuned Ensemble 2)
- **Stacking Ensemble** (Champion Model: RF + GB with Linear Regression meta-learner)

In [ ]:
from src.train import train_and_evaluate_all
results, metadata = train_and_evaluate_all()
pd.DataFrame(results).T[['train_r2', 'test_r2', 'overfit_gap_r2', 'test_mae', 'test_rmse']]

## 4. Step 7: Granular Error Analysis Across AQI Severity Categories

In [ ]:
from src.error_analysis import perform_error_analysis
report = perform_error_analysis()
pd.DataFrame(report['performance_by_category']).T

## 5. Health Advisory Generation

In [ ]:
from src.advisory import get_aqi_advisory
for sample_aqi in [35, 75, 140, 240, 360, 450]:
    adv = get_aqi_advisory(sample_aqi)
    print(f"AQI {sample_aqi} -> [{adv['category']}] {adv['summary']}")